# Packages

In [1]:
!pip install "gymnasium[atari]" "gymnasium[accept-rom-license]"

In [2]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from collections import deque
import ale_py
import numpy as np

In [3]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model

In [4]:
class QNetwork(nn.Module):
  def __init__(self,num_actions):
    super(QNetwork,self).__init__()
    self.conv1=nn.Conv2d(in_channels=4, out_channels=32, kernel_size=4, stride=2)
    self.conv2=nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2)
    self.conv3=nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=2)

    self.linear1=nn.Linear(in_features=5184, out_features=512) ## 5184 is calculated value
    self.linear2=nn.Linear(in_features=512, out_features=num_actions)


  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=F.relu(self.conv2(x))
    x=F.relu(self.conv3(x))
    x=torch.flatten(x,start_dim=1)
    x=F.relu(self.linear1(x))
    x=self.linear2(x)
    return x


In [32]:
class SpaceInvader_DQN(object):
  lr=1e-4
  replay_mem_size=50000
  min_batch_size=32
  gamma=0.99
  loss_fn=nn.HuberLoss()
  optim=None
  network_sync_rate=1000

  def episode(self, batch, policy_dqn, target_dqn):
    states, actions, rewards, next_states, dones=zip(*batch)
    states = torch.stack([torch.as_tensor(np.array(s)) for s in states]).to(device) / 255.0
    next_states = torch.stack([torch.as_tensor(np.array(s)) for s in next_states]).to(device) / 255.0
    actions=torch.tensor(actions).unsqueeze(1).to(device)
    rewards=torch.tensor(rewards, dtype=torch.float).unsqueeze(1).to(device)
    dones=torch.tensor(dones, dtype=torch.float).unsqueeze(1).to(device)

    with torch.no_grad():
      next_q_values=target_dqn(next_states)
      max_next_q_val, _=torch.max(next_q_values, dim=1, keepdim=True)
      max_target_val=self.gamma*max_next_q_val
      target_val=max_target_val*(1-dones)
      target_q_val=rewards+target_val

    q_vals=policy_dqn(states).gather(1, actions)
    loss=self.loss_fn(q_vals, target_q_val)
    self.optim.zero_grad()
    loss.backward()
    self.optim.step()
    return loss.item()

  def train(self,episodes, render=False):
    env=gym.make('SpaceInvadersNoFrameskip-v4', render_mode='human' if render else None)
    env=AtariPreprocessing(env,screen_size=84, grayscale_obs=True, scale_obs=False)
    env=FrameStackObservation(env, stack_size=4)
    action_space=env.action_space.n
    self.policy_dqn=QNetwork(action_space).to(device)
    self.target_dqn=QNetwork(action_space).to(device)
    self.target_dqn.load_state_dict(self.policy_dqn.state_dict())
    self.optim=optim.Adam(self.policy_dqn.parameters(), lr=self.lr)
    replay_mem=deque(maxlen=self.replay_mem_size)
    self.loss_list=[]
    min_eps=0.01
    eps_decay=0.995
    eps=1.0
    steps=0
    loss=0

    for i in range(episodes):
      states=env.reset()[0]
      terminated=False
      truncated=False
      while(not terminated and not truncated):
        rand=random.random()
        if(rand<eps):
          action=env.action_space.sample()
        else:
          with torch.no_grad():
            states_norm=torch.tensor(states, dtype=torch.float).unsqueeze(0).to(device)/255.0
            action=self.policy_dqn(states_norm).argmax().item()

        new_state, reward, terminated, truncated, _=env.step(action)
        clipped_reward = max(-1.0, min(reward, 1.0))
        replay_mem.append((states, action, clipped_reward, new_state, terminated))
        states=new_state
        steps+=1
        if len(replay_mem)>self.min_batch_size:
          min_batch=random.sample(replay_mem, self.min_batch_size)
          loss=self.episode(min_batch, self.policy_dqn, self.target_dqn)
          self.loss_list.append(loss)

        if steps>=self.network_sync_rate:
          self.target_dqn.load_state_dict(self.policy_dqn.state_dict())
          steps=0

      eps=max(min_eps, eps_decay*eps)
      if(i%10==0):
        print(f"Step: {i}, loss: {loss}, epsilon={eps}")


In [33]:
agent=SpaceInvader_DQN()
agent.train(episodes=100, render=False)

Step: 0, loss: 0.0003892732784152031, epsilon=0.995
Step: 10, loss: 0.005548608954995871, epsilon=0.946354579813443
Step: 20, loss: 0.00276110228151083, epsilon=0.9000874278732445
Step: 30, loss: 0.008733071386814117, epsilon=0.8560822709551227
Step: 40, loss: 0.018182052299380302, epsilon=0.8142285204175609
Step: 50, loss: 0.02205565571784973, epsilon=0.7744209942832988
Step: 60, loss: 0.02192116156220436, epsilon=0.736559652908221
Step: 70, loss: 0.01767704263329506, epsilon=0.7005493475733617
Step: 80, loss: 0.012394046410918236, epsilon=0.6662995813682115
Step: 90, loss: 0.0156303308904171, epsilon=0.6337242817644086


In [36]:
def save_gif(agent, filename="space_invaders.gif"):
    env = gym.make('SpaceInvadersNoFrameskip-v4', render_mode='rgb_array')
    env = AtariPreprocessing(env, screen_size=84, grayscale_obs=True, scale_obs=False)
    env = FrameStackObservation(env, stack_size=4)
    state, _ = env.reset()
    done = False
    frames = []

    while not done:
        frame = env.render()
        frames.append(frame)

        with torch.no_grad():
            # Add the missing normalization!
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device) / 255.0
            action = agent.policy_dqn(state_tensor).argmax().item()

        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    env.close()

    print(f"Saving {len(frames)} frames to {filename}...")
    imageio.mimsave(filename, frames, fps=30)
    print("Done!")

In [37]:
save_gif(agent)

Saving 905 frames to space_invaders.gif...
Done!
